# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [2]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.0 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.18.9-py3-none-any.whl size=1820406 sha256=1e549a3fc648c50f7dc2a64960548c32a40883c72fd4811439e58850a16437d6
  Stored in directory: /root/.cache/pip/wheels/14/ff/77/6df576c7f26969435706ee20282924bef2c06ec6763192d0ab
Successfully built deepspeed
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: dat

### Data Preparation

In [3]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: gchhablani/bert-base-cased-finetuned-qqp
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [5]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [6]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [7]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [8]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=64, shuffle=False, collate_fn=transformers.default_data_collator,num_workers=2,
)
model.to(device)
sum_correct = 0
with torch.no_grad():
  for batch in val_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    logits = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    ).logits
    preds = torch.argmax(logits, dim = 1)
    correct = (preds == batch['labels']).sum()
    sum_correct += correct



accuracy = sum_correct/len(val_set)

In [14]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [15]:
from transformers import TrainingArguments, Trainer

In [108]:
tokenizer = transformers.AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
model = transformers.AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
)
model.to(device)

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=128,
        truncation=True,
    )
    result["label"] = examples["label"]
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True, remove_columns=["idx", "label_text"])
qqp_preprocessed = qqp_preprocessed.remove_columns(["text1", "text2", "token_type_ids"])

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

In [6]:
def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=128,
        truncation=True,
    )
    result["label"] = examples["label"]
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True, remove_columns=["idx", "label_text"])
qqp_preprocessed = qqp_preprocessed.remove_columns(["text1", "text2", "token_type_ids"])

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [109]:
import numpy as np
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = logits.astype(np.float32)
    preds = np.argmax(logits, axis=1)
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

In [110]:
training_args = TrainingArguments(
    output_dir="./deberta",
    eval_strategy="steps",
    eval_steps=1000,
    logging_steps=1000,
    logging_dir="./logs",
    report_to="none",
    learning_rate=1e-5,
    num_train_epochs=4,
    warmup_ratio=0.1,
    gradient_accumulation_steps=2,
    lr_scheduler_type="linear",
    per_device_train_batch_size=12,
    per_device_eval_batch_size=32,
    max_grad_norm=1.0,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [111]:
small_train = qqp_preprocessed["train"].select(range(20000))

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = small_train,
    eval_dataset = qqp_preprocessed['validation'],
    processing_class= tokenizer,
    compute_metrics = compute_metrics
)

In [112]:
model.train()
batch = next(iter(trainer.get_train_dataloader()))
batch = {k: v.to(device) for k, v in batch.items()}
print("Batch keys:", batch.keys())
print("Labels:", batch["labels"][:5])

output = model(**batch)
print("Loss:", output.loss)
print("Logits:", output.logits[:3])

Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Labels: tensor([1, 0, 1, 0, 0], device='cuda:0')
Loss: tensor(0.7202, device='cuda:0', grad_fn=<NllLossBackward0>)
Logits: tensor([[-0.1575, -0.0783],
        [-0.0937, -0.0409],
        [-0.1662, -0.0352]], device='cuda:0', grad_fn=<SliceBackward0>)


In [113]:
print(training_args.fp16)
print(training_args.bf16)

False
False


In [114]:
model.eval()
eval_loader = trainer.get_eval_dataloader()
batch = next(iter(eval_loader))
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    output = model(**batch)

print("Loss:", output.loss)
print("Logits dtype:", output.logits.dtype)
print("Logits:", output.logits[:3])
print("Labels:", batch["labels"][:5])

Loss: tensor(0.7177, device='cuda:0')
Logits dtype: torch.float32
Logits: tensor([[-0.1271, -0.0088],
        [-0.1265, -0.0065],
        [-0.1284, -0.0098]], device='cuda:0')
Labels: tensor([0, 0, 1, 0, 0], device='cuda:0')


In [115]:
print(qqp_preprocessed["train"].column_names)

['label', 'input_ids', 'attention_mask']


In [116]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss,Validation Loss,Accuracy
1000,0.815907,0.302167,0.870715
2000,0.484116,0.332552,0.878011
3000,0.347956,0.391214,0.878407


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Accuracy
1000,0.815907,0.302167,0.870715
2000,0.484116,0.332552,0.878011
3000,0.347956,0.391214,0.878407
3336,0.347956,0.385522,0.878506


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3336, training_loss=0.5243141073688901, metrics={'train_runtime': 4629.9074, 'train_samples_per_second': 17.279, 'train_steps_per_second': 0.721, 'total_flos': 5262315479040000.0, 'train_loss': 0.5243141073688901, 'epoch': 4.0})

In [118]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/deberta-qqp")
tokenizer.save_pretrained("/content/drive/MyDrive/deberta-qqp")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/deberta-qqp/tokenizer_config.json',
 '/content/drive/MyDrive/deberta-qqp/tokenizer.json')

In [5]:
from google.colab import drive

drive.mount('/content/drive')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = transformers.AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/deberta-qqp",
    torch_dtype=torch.float32
)
tokenizer = transformers.AutoTokenizer.from_pretrained("/content/drive/MyDrive/deberta-qqp")

model.to(device)
model.eval()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [10]:
def find_duplicates(question, train_set, model, tokenizer, top_k=5, batch_size=64):
    questions = train_set["text1"][:1000]
    scores = []

    for i in range(0, len(questions), batch_size):
        batch_questions = questions[i:i+batch_size]
        tokens = tokenizer(
            [question] * len(batch_questions),
            batch_questions,
            padding="max_length",
            max_length=128,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
          logits = model(**tokens).logits
          probs = torch.softmax(logits, dim=1)[:, 1]
        scores.extend(probs.cpu().tolist())
    top5_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [(questions[i], scores[i]) for i in top5_idx]

In [11]:
print(next(model.parameters()).device)

cuda:0


In [12]:
find_duplicates("How do I become a better programmer?", qqp["train"], model, tokenizer)

[('How can I learn programming from scratch?', 0.7920785546302795),
 ('What are the things I need to learn hacking?', 0.2310233861207962),
 ('What are the best online coding bootcamps?', 0.10882988572120667),
 ('How shall I become a software developer in India?', 0.055265843868255615),
 ('What is the best way to learn and practice C programming?',
  0.03855372965335846)]

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(qqp["train"]["text1"])

def fast_find_duplicates(question, train_set, model, tokenizer, top_k=5):
    query_vec = vectorizer.transform([question])
    similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
    top100_idx = similarities.argsort()[::-1][:100]
    candidate_questions = [train_set["text1"][i] for i in top100_idx]

    tokens = tokenizer(
        [question] * len(candidate_questions),
        candidate_questions,
        padding="max_length",
        max_length=128,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**tokens).logits
        probs = torch.softmax(logits, dim=1)[:, 1]

    top5_idx = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)[:top_k]
    return [(candidate_questions[i], probs[i].item()) for i in top5_idx]

In [14]:
find_duplicates("How do I become a better programmer?", qqp["train"], model, tokenizer)

[('How can I learn programming from scratch?', 0.7920785546302795),
 ('What are the things I need to learn hacking?', 0.2310233861207962),
 ('What are the best online coding bootcamps?', 0.10882988572120667),
 ('How shall I become a software developer in India?', 0.055265843868255615),
 ('What is the best way to learn and practice C programming?',
  0.03855372965335846)]

In [15]:
import time

question = "How do I become a better programmer?"

start = time.time()
result1 = find_duplicates(question, qqp["train"], model, tokenizer)
slow_time = time.time() - start
print(f"Slow: {slow_time:.2f}s")

start = time.time()
result2 = fast_find_duplicates(question, qqp["train"], model, tokenizer)
fast_time = time.time() - start
print(f"Fast: {fast_time:.2f}s")

print(f"Speedup: {slow_time/fast_time:.1f}x")

Slow: 10.60s
Fast: 1.32s
Speedup: 8.0x
